In [9]:
import logging
import re
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
import pandas as pd
import soccerdata as sd


from fpl_project.scripts import config
from fpl_project.scripts.config import SEASONS

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(config.DATA_DIR / "fpl_historical.log", mode="w", encoding="utf-8")
    ]
)
logger = logging.getLogger(__name__)




In [11]:
config = ContentConfig()

In [12]:
class FetchData(ABC):
    def __init__(self, seasons: list[str]):
        self.seasons = seasons

    @abstractmethod
    def _merge_and_clean(self) -> pd.DataFrame:
        pass

    @abstractmethod
    def get_data(self) -> pd.DataFrame:
        pass

    


In [ ]:
class FetchVaastav(FetchData):
    def __init__(self, config_data: ContentConfig):
        # Vaastav covers only fully archived seasons — the two most recent
        # seasons (last completed + current ongoing) are handled by FetchFCI.
        super().__init__(seasons=config_data.SEASONS[:-2])
        self.config = config_data

    def _fetch_gameweeks(self) -> pd.DataFrame:
        frames = []
        skipped = 0

        for season in self.seasons:
            season_frames = []
            for gw in range(1, 39):
                try:
                    url = self.config.VAASTAV_GW_URL.format(season, gw)
                    df = pd.read_csv(url)
                    if not df.empty:
                        season_frames.append(
                            df.assign(season=season, gw=gw).dropna(axis=1, how="all")
                        )
                except Exception as e :
                    skipped += 1

            if season_frames:
                frames.extend(season_frames)
                logger.info("Fetched GW data for season %s. Number of gameweeks: %d", season, len(season_frames))
            else:
                logger.warning("No GW data found for season %s.", season)

        if skipped > 0:
            logger.warning("Skipped a total of %d GW files (file not found on server or network error).", skipped)

        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    def _fetch_ids(self) -> pd.DataFrame:
        frames = []

        for season in self.seasons:
            try:
                url = self.config.VAASTAV_ID_URL.format(season)
                df = pd.read_csv(
                    url,
                    usecols=["id", "first_name", "second_name", "code"]
                )
                df["season"] = season
                df["name"] = df["first_name"] + " " + df["second_name"]
                frames.append(df.drop(columns=["first_name", "second_name"]))
                logger.info("Fetched player metadata for season %s. Number of records: %d", season, len(df))
            except Exception as e:
                logger.error("Error fetching player metadata for season %s: %s", season, e)

        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    def _merge_and_clean(self, gw_df: pd.DataFrame, id_df: pd.DataFrame) -> pd.DataFrame:
        if gw_df.empty or id_df.empty:
            logger.error("Cannot perform merge operation. One of the input datasets is empty.")
            return pd.DataFrame()

        logger.info("Started merging GW and IDs datasets.")

        df = pd.merge(
            gw_df, id_df,
            how="inner",
            left_on=["element", "season"],
            right_on=["id", "season"]
        ).drop(columns=["id"])

        if "name_y" in df.columns:
            df.drop(columns=["name_y"], inplace=True)
        if "name_x" in df.columns:
            df.rename(columns={"name_x": "name"}, inplace=True)

        if "value" in df.columns:
            df["value"] = df["value"] / 10

        df["selected"] = df.groupby(["season", "gw"])["selected"].transform(
            lambda x: x / x.sum()
        )

        df["kickoff_time"] = (
            df["kickoff_time"]
            .astype(str)
            .str.replace("Z", "", regex=False)
            .str.slice(0, 19)
        )
        kickoff_dt = pd.to_datetime(df['kickoff_time'], errors='coerce')

        df['kickoff_time'] = pd.to_datetime(kickoff_dt.dt.strftime('%d/%m/%Y'), dayfirst=True)

        df["season"] = df["season"].apply(
            lambda x: re.sub(r'^20(\d{2})-(\d{2})$', r'\1\2', str(x))
        )

        df = df.drop(columns=self.config.DROP_COLS, errors="ignore")

        before = len(df)
        df = df.drop_duplicates(subset=["code", "season", "gw"])
        dropped = before - len(df)

        if dropped > 0:
            logger.warning(
                "Removed %d duplicate rows (subset: code, season, gw).",
                dropped
            )

        logger.info("Cleaning process completed. Final dataset dimensions: %d rows, %d columns.", *df.shape)

        return df

    def get_data(self) -> pd.DataFrame:
        logger.info("Started full data fetching and processing pipeline from the Vaastav repository.")

        gw_df = self._fetch_gameweeks()
        id_df = self._fetch_ids()

        df = self._merge_and_clean(gw_df, id_df)
        return df

    


    

In [14]:
class FetchFCI(FetchData):
    """FCI only works for the current, ongoing season (no archive)."""

    def __init__(self, config_data: ContentConfig):
        current_season = config_data.SEASONS[-1]
        super().__init__(seasons=[current_season])
        self.config = config_data
        self.current_season = current_season
        yr1, yr2 = current_season.split("-")
        self.fci_season_path = f"{yr1}-20{yr2}"  # "2026-27" -> "2026-2027"

    def _get_fci_base(self) -> tuple[pd.DataFrame, pd.DataFrame]:
        try:
            playerstats_url = self.config.FCI_PLAYERSTATS_URL.format(self.fci_season_path)
            playerstats = pd.read_csv(playerstats_url, low_memory=False)

            teams_url = playerstats_url.replace("playerstats.csv", "teams.csv")
            teams_df = pd.read_csv(teams_url, usecols=["id", "code", "name"])
            teams_df = teams_df.rename(columns={"name": "team", "id": "team_id", "code": "team_code"})

            return playerstats, teams_df
        except Exception as e:
            logger.error("Error fetching base FCI data for season %s: %s", self.current_season, e)
            return pd.DataFrame(), pd.DataFrame()

    def _fetch_gameweeks(self) -> tuple[pd.DataFrame, pd.DataFrame]:
        players_frames, matches_frames = [], []
        skipped = 0

        for gw in range(1, 39):
            try:
                url_players = self.config.FCI_URL.format(self.fci_season_path, gw, "players")
                df_p = pd.read_csv(
                    url_players, low_memory=False,
                    usecols=["player_id", "first_name", "second_name", "position", "player_code", "team_code"]
                )
                players_frames.append(df_p)

                url_matches = self.config.FCI_URL.format(self.fci_season_path, gw, "matches")
                df_m = pd.read_csv(
                    url_matches, low_memory=False,
                    usecols=["match_id", "gameweek", "kickoff_time", "home_team", "away_team", "home_score", "away_score"]
                )
                matches_frames.append(df_m)
            except Exception:
                skipped += 1

        if skipped > 0:
            logger.warning("Skipped %d FCI files for season %s (missing GW or network error).", skipped, self.current_season)

        if not players_frames or not matches_frames:
            return pd.DataFrame(), pd.DataFrame()

        players = pd.concat(players_frames, ignore_index=True).drop_duplicates(subset=["player_id"])
        matches = pd.concat(matches_frames, ignore_index=True)
        return players, matches

    def _merge_and_clean(self) -> pd.DataFrame:
        players, matches = self._fetch_gameweeks()
        playerstats, teams_df = self._get_fci_base()

        if players.empty or matches.empty or playerstats.empty or teams_df.empty:
            logger.error("Cannot perform FCI merge. One of the input datasets is empty.")
            return pd.DataFrame()

        players["name"] = players["first_name"] + " " + players["second_name"]
        players = players.drop(columns=["first_name", "second_name"])
        players = pd.merge(players, teams_df, on="team_code", how="left")

        df = pd.merge(playerstats, players, how="left", left_on="id", right_on="player_id")
        df = pd.merge(df, matches, how="left", left_on="gw", right_on="gameweek")

        df = df[(df["team_id"] == df["home_team"]) | (df["team_id"] == df["away_team"])].copy()
        df["was_home"] = df["team_id"] == df["home_team"]

        for col in ["transfers_in_event", "transfers_out_event"]:
            df[col] = df[col].fillna(0)
        df["transfers_balance_event"] = df["transfers_in_event"] - df["transfers_out_event"]

        df["position"] = df["position"].map({
            "Defender": "DEF", "Forward": "FWD", "Goalkeeper": "GK", "Midfielder": "MID"
        })

        df = df[df["match_id"].astype(str).str.contains("prem", na=False)]

        rename_dict = {
            "id": "element", "ep_this": "xP", "match_id": "fixture", "now_cost": "value",
            "selected_by_percent": "selected", "home_score": "team_h_score",
            "away_score": "team_a_score", "player_code": "code",
        }
        df = df.rename(columns=rename_dict)
        df["round"] = df["gw"]

        try:
            df = df[self.config.FCI_TARGET_COLUMNS].copy()
        except Exception as e:
            logger.error("Missing required FCI columns: %s", e)
            return pd.DataFrame()

        df = df.rename(columns={
            "transfers_out_event": "transfers_out",
            "transfers_balance_event": "transfers_balance",
            "transfers_in_event": "transfers_in",
            "event_points": "total_points",
        })

        logger.info("FCI cleaning process completed for season %s. Dimensions: %d rows, %d columns.",
                    self.current_season, *df.shape)

        return df.assign(season=self.current_season)

    def get_data(self) -> pd.DataFrame:
        logger.info("Started fetching data from FPL-Core-Insights (FCI) for season %s.", self.current_season)
        return self._merge_and_clean()

    


In [15]:
class FetchBetting(FetchData):
    def __init__(self, config_data: ContentConfig):
        super().__init__(seasons=config_data.BETTING_SEASONS)
        self.config = config_data
        self.teams_map = {
            'Arsenal': 'Arsenal', 
            'Aston Villa': 'Aston Villa', 
            'Bournemouth': 'Bournemouth', 
            'Brentford': 'Brentford',
            'Brighton': 'Brighton', 
            'Burnley': 'Burnley',
            'Crystal Palace': 'Crystal Palace', 
            'Everton': 'Everton', 
            'Fulham': 'Fulham', 
            'Ipswich': 'Ipswich',
            'Leeds': 'Leeds', 
            'Leicester': 'Leicester', 
            'Liverpool': 'Liverpool',
            'Luton': 'Luton', 
            "Hull" : "Hull",
            'Man City': 'Man City', 
            'Man United': 'Man Utd', 
            'Newcastle': 'Newcastle',
            "Nott'm Forest": "Nott'm Forest", 
            'Sheffield United': 'Sheffield Utd',
            'Southampton': 'Southampton', 
            'Sunderland': 'Sunderland', 
            'Tottenham': 'Spurs', 
            'West Ham': 'West Ham',
            'Wolves': 'Wolves', 
            'Chelsea': 'Chelsea'
        }

        self.betting_features = [
            'date', 'time', 'hometeam', 'awayteam', 'season',
            'avgh', 'avgd', 'avga',
            'avg>2.5', 'avg<2.5',
            'ahh'
        ]

    def _fetch_seasons(self):
        frames = []
        url = self.config.BETTING_URL
        for season in self.seasons:
            try:
                df = pd.read_csv(url.format(season))
                df["season"] = season
                frames.append(df)
            except Exception as e:
                logger.error("Error fetching betting data for season %s: %s", season, e)
        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    def _merge_and_clean(self) -> pd.DataFrame:
        df = self._fetch_seasons()
        if df.empty:
            return df

        df.columns = df.columns.str.lower()
        df = df.drop(columns=["div", "referee"], errors="ignore")

        df.hometeam, df.awayteam = df.hometeam.map(self.teams_map), df.awayteam.map(self.teams_map)

        available_features = [c for c in self.betting_features if c in df.columns]
        df = df[available_features]

        df.columns = df.columns.str.replace(">", "_over_", regex=False).str.replace("<", "_under_", regex=False)
        df.date = pd.to_datetime(df.date, dayfirst=True)
        return df

    def _odds_to_probs(self) -> pd.DataFrame:
        df = self._merge_and_clean()
        if df.empty:
            return df

        match_market_groups = [['avgh', 'avgd', 'avga']]
        goals_market_groups = [['avg_over_2.5', 'avg_under_2.5']]

        for group in match_market_groups:
            if all(col in df.columns for col in group):
                raw_probs = 1 / df[group]
                total_margin = raw_probs.sum(axis=1)
                prob_names = [col.replace('avg', 'prob') for col in group]
                df[prob_names] = raw_probs.div(total_margin, axis=0)

        for group in goals_market_groups:
            if all(col in df.columns for col in group):
                raw_probs = 1 / df[group]
                total_margin = raw_probs.sum(axis=1)
                prob_names = [col.replace('avg', 'prob') for col in group]
                df[prob_names] = raw_probs.div(total_margin, axis=0)

        logger.info("Successfully converted betting odds into normalized probabilities.")
        return df

    def _transform_wide_to_long(self, df_wide: pd.DataFrame) -> pd.DataFrame:
        if df_wide.empty:
            return df_wide

        home_df = pd.DataFrame()
        home_df['date'] = df_wide['date']
        home_df['season'] = df_wide['season']
        home_df['team'] = df_wide['hometeam']
        home_df['opponent_team'] = df_wide['awayteam']
        home_df['was_home'] = True

        home_df['prob_win'] = df_wide['probh']
        home_df['prob_draw'] = df_wide['probd']
        home_df['prob_lose'] = df_wide['proba']

        if 'ahh' in df_wide.columns:
            home_df['ahh'] = df_wide['ahh']

        away_df = pd.DataFrame()
        away_df['date'] = df_wide['date']
        away_df['season'] = df_wide['season']
        away_df['team'] = df_wide['awayteam']
        away_df['opponent_team'] = df_wide['hometeam']
        away_df['was_home'] = False

        away_df['prob_win'] = df_wide['proba']
        away_df['prob_draw'] = df_wide['probd']
        away_df['prob_lose'] = df_wide['probh']

        if 'ahh' in df_wide.columns:
            away_df['ahh'] = -df_wide['ahh']

        for df_part in [home_df, away_df]:
            for col in ['prob_over_2.5', 'prob_under_2.5']:
                if col in df_wide.columns:
                    df_part[col] = df_wide[col]

        df_long = pd.concat([home_df, away_df], ignore_index=True)
        df_long = df_long.sort_values(by=['season', 'date', 'team']).reset_index(drop=True)

        return df_long

    def get_data(self) -> pd.DataFrame:
        df_wide = self._odds_to_probs()
        df_long = self._transform_wide_to_long(df_wide)

        output_cols = ['date', 'season', 'team', 'opponent_team', 'was_home',
                       'prob_win', 'prob_draw', 'prob_lose', 'ahh',
                       'prob_over_2.5', 'prob_under_2.5']

        final_cols = [c for c in output_cols if c in df_long.columns]

        return df_long[final_cols]

    

In [ ]:
class MergeData:
    def __init__(self, vaastav_df: pd.DataFrame,
                 betting_df: pd.DataFrame,
                 elo_df: pd.DataFrame = None):

        self.vaastav_df = vaastav_df
        self.betting_df = betting_df
        self.elo_df = elo_df

    # def _elo_betting(self) -> pd.DataFrame:
    #     elo_df = self.elo_df
    #     betting_df = self.betting_df
    #
    #     if elo_df.empty or betting_df.empty:
    #         logger.warning("Missing ELO or betting data — skipping ELO+betting merge.")
    #         return pd.DataFrame()
    #
    #     betting_df = betting_df.sort_values("date")
    #     elo_df = elo_df.sort_values("from")
    #
    #     betting_elo_df = pd.merge_asof(
    #         betting_df,
    #         elo_df,
    #         left_on="date",
    #         right_on="from",
    #         by="team",
    #         direction="backward"
    #     )
    #
    #     if "from" in betting_elo_df.columns:
    #         betting_elo_df = betting_elo_df.drop(columns=['from'])
    #     if "to" in betting_elo_df.columns:
    #         betting_elo_df = betting_elo_df.drop(columns=['to'])
    #
    #     return betting_elo_df

    def _elo_betting_vaastav(self) -> pd.DataFrame:
        vaastav_df = self.vaastav_df
        # elo_betting_df = self._elo_betting()
        elo_betting_df = self.betting_df  # ELO merge disabled — using betting_df as-is

        if vaastav_df.empty or elo_betting_df.empty:
            return pd.DataFrame()

        vaastav_df = vaastav_df.copy()
        elo_betting_df = elo_betting_df.copy()

        vaastav_df['kickoff_time'] = pd.to_datetime(vaastav_df['kickoff_time'])
        elo_betting_df['date'] = pd.to_datetime(elo_betting_df['date'])

        vaastav_df['team'] = vaastav_df['team'].astype(str)
        elo_betting_df['team'] = elo_betting_df['team'].astype(str)
        vaastav_df['season'] = vaastav_df['season'].astype(str)
        elo_betting_df['season'] = elo_betting_df['season'].astype(str)

        df = pd.merge(
            vaastav_df,
            elo_betting_df,
            left_on=['kickoff_time', 'team', 'season'],
            right_on=['date', 'team', 'season'],
            how='inner'
        )
        df.drop(columns=['was_home_x', 'opponent_team_x'], inplace=True)
        df.rename(columns={'was_home_y': 'was_home',
                           'opponent_team_y': 'opponent_team'}, inplace=True)

        return df

    def get_data(self) -> pd.DataFrame:
        df = self._elo_betting_vaastav()
        return df


if __name__ == "__main__":
    logger.info("Starting FPL data processing pipeline.")

    content_cfg = ContentConfig()
    vaastav_fetcher = FetchVaastav(config_data=content_cfg)
    betting_fetcher = FetchBetting(config_data=content_cfg)
    fci_fetcher = FetchFCI(config_data=content_cfg)

    vaastav_df = vaastav_fetcher.get_data()
    fci_df = fci_fetcher.get_data()

    if not fci_df.empty:
        fci_df["kickoff_time"] = pd.to_datetime(
            pd.to_datetime(fci_df["kickoff_time"], errors="coerce").dt.strftime("%d/%m/%Y"),
            dayfirst=True
        )
        # "fixture" is a numeric ID in vaastav_df but a string match_id in fci_df
        # (e.g. "25-26-prem-tottenham-hotspur-vs-manchester-united") — unify the
        # dtype before concatenating, otherwise the mixed-type object column
        # breaks pyarrow's schema inference in to_parquet later.
        if "fixture" in vaastav_df.columns:
            vaastav_df["fixture"] = vaastav_df["fixture"].astype(str)
        if "fixture" in fci_df.columns:
            fci_df["fixture"] = fci_df["fixture"].astype(str)
        vaastav_df = pd.concat([vaastav_df, fci_df], axis=0, ignore_index=True)
        vaastav_df["season"] = vaastav_df["season"].apply(
            lambda x: re.sub(r'^20(\d{2})-(\d{2})$', r'\1\2', str(x))
        )

    betting_df = betting_fetcher.get_data()
    # elo_fetcher = FetchELO(config_data=content_cfg, betting_df=betting_df)
    # elo_df = elo_fetcher.get_data()

    merger = MergeData(vaastav_df=vaastav_df, betting_df=betting_df)
    data = merger.get_data()

    if not data.empty:
        data.to_parquet(config.FPL_VAASTAV_PATH, index=False)
        logger.info("Successfully exported dataset to Parquet format: %s", config.FPL_VAASTAV_PATH)
    else:
        logger.error("Pipeline stopped. Resulting dataset (final_data) is empty.")

    logger.info("Process finished.")

In [ ]:






# --- ELO logic disabled ---
# class FetchELO(FetchData):
#     def __init__(self, config_data: ContentConfig, betting_df: pd.DataFrame):
#         super().__init__(seasons=config_data.BETTING_SEASONS)
#         self.config = config_data
#         self.betting_df = betting_df
#
#         self.betting_to_elo_map = {
#             'Spurs': 'Tottenham',
#             'Man Utd': 'Man United',
#             "Nott'm Forest": 'Forest',
#             'Sheffield Utd': 'Sheffield United'
#         }
#         self.elo_to_betting_map = {v: k for k, v in self.betting_to_elo_map.items()}
#
#     def _fetch_seasons(self) -> pd.DataFrame:
#         if self.betting_df.empty:
#             return pd.DataFrame()
#         self.betting_df.date = pd.to_datetime(self.betting_df.date, dayfirst=True)
#         min_date = self.betting_df.date.min() - pd.Timedelta(days=100)
#
#         elo_teams_list = [self.betting_to_elo_map.get(team, team) for team in self.betting_df['team'].unique()]
#         elo = sd.ClubElo()
#         frames = []
#         for team in elo_teams_list:
#             try:
#                 team_history = elo.read_team_history(team)
#                 team_history = team_history[team_history.index >= min_date]
#                 team_history = team_history.reset_index()
#                 frames.append(team_history)
#             except Exception as e:
#                 logger.error(f"Error fetching Elo data for team {team}: {e}")
#
#         if frames:
#             return pd.concat(frames, axis=0, ignore_index=True)
#         else:
#             return pd.DataFrame()
#
#     def _merge_and_clean(self) -> pd.DataFrame:
#         df = self._fetch_seasons()
#         if df.empty:
#             return pd.DataFrame()
#
#         df.columns = df.columns.str.lower()
#
#         df["from"] = pd.to_datetime(df["from"])
#         df["to"] = pd.to_datetime(df["to"])
#
#         df['team'] = df['team'].replace(self.elo_to_betting_map)
#
#         target_cols = ["from", "to", "team", "elo"]
#         available_cols = [c for c in target_cols if c in df.columns]
#         df = df[available_cols]
#
#         if df.empty:
#             return pd.DataFrame()
#         df = df.sort_values(by=['team', 'from']).reset_index(drop=True)
#
#         return df
#
#     def get_data(self) -> pd.DataFrame:
#         df = self._merge_and_clean()
#         return df
#
#     

